# Bước 1 — Đọc tài liệu
Notebook đọc PDF, DOCX, TXT và Excel. Chạy từng cell từ trên xuống. Kết quả cuối cùng là `storage/documents.json`.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from docx import Document
from pypdf import PdfReader

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_DIR = PROJECT_DIR / 'dataset'
STORAGE_DIR = PROJECT_DIR / 'storage'
print('Project:', PROJECT_DIR)
print('Số file:', len(list(DATASET_DIR.iterdir())))

## Hàm đọc từng loại file
Mỗi hàm trả về danh sách phần nội dung cùng vị trí trang, sheet hoặc document.

In [ ]:
def read_pdf(path):
    return [{'content': page.extract_text() or '', 'location': f'page_{i}'}
            for i, page in enumerate(PdfReader(path).pages, 1) if (page.extract_text() or '').strip()]

def read_docx(path):
    doc = Document(path)
    parts = [p.text for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            parts.append(' | '.join(cell.text.strip() for cell in row.cells))
    return [{'content': '\n'.join(parts), 'location': 'document'}]

def read_txt(path):
    return [{'content': path.read_text(encoding='utf-8'), 'location': 'document'}]

def read_excel(path):
    results = []
    book = pd.ExcelFile(path)
    for sheet in book.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet).fillna('')
        lines = [' | '.join(map(str, df.columns))]
        lines += [' | '.join(map(str, row)) for row in df.values]
        results.append({'content': '\n'.join(lines), 'location': f'sheet:{sheet}'})
    return results

READERS = {'.pdf': read_pdf, '.docx': read_docx, '.txt': read_txt, '.xlsx': read_excel}

In [ ]:
documents = []
for path in sorted(DATASET_DIR.iterdir()):
    reader = READERS.get(path.suffix.lower())
    if not reader:
        continue
    for item in reader(path):
        documents.append({**item, 'source': path.name, 'file_type': path.suffix[1:].lower()})
    print('Đã đọc:', path.name)

print('Tổng phần tài liệu:', len(documents))

## Quan sát dữ liệu trước khi lưu

In [ ]:
documents[0]

In [ ]:
STORAGE_DIR.mkdir(exist_ok=True)
output = STORAGE_DIR / 'documents.json'
output.write_text(json.dumps(documents, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', output)